### Instalação das dependências

Os pacotes abaixo são necessários para treinar e comparar o modelo novo com o modelo atual em produção. Execute esta célula antes de qualquer outra.

In [ ]:
!pip install scikit-learn pandas joblib -q

# Código 3.3 — Workflow Seguro de Retreinamento Automatizado

**Objetivo:** Demonstrar a orquestração do retreinamento de um modelo, garantindo que o retreinamento só substitua a versão atual caso comprove desempenho superior.

### Passo 1: Leitura das Novas Coletas e Retreinamento do Modelo

Carregamos a base atualizada para treinar uma nova versão do pipeline.

In [ ]:
from pathlib import Path
import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Simular novos dados acumulados
df = pd.read_csv("dados_churn.csv")
X = df.drop(columns=["churn"])
y = df["churn"]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=100, stratify=y)

# Treinar nova versão (Candidato v2)
novo_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier(n_estimators=120, max_depth=8, random_state=42))
])

novo_pipeline.fit(X_train, y_train)
print("Novo modelo v2 treinado com dados atualizados!")

### Passo 2: Comparação Segura e Substituição de Artefato

Comparamos as métricas e atualizamos o arquivo de produção apenas sob aprovação.

In [ ]:
from sklearn.metrics import roc_auc_score

PATH_PRODUCAO = Path("models/champion.joblib")

# Carregar modelo atual de produção
modelo_atual = joblib.load(PATH_PRODUCAO)

# Avaliar ambos na mesma base de validação
auc_atual = roc_auc_score(y_val, modelo_atual.predict_proba(X_val)[:, 1])
auc_novo = roc_auc_score(y_val, novo_pipeline.predict_proba(X_val)[:, 1])

print(f"AUC Modelo Atual  : {auc_atual:.4f}")
print(f"AUC Novo Modelo v2: {auc_novo:.4f}")

# Critério de substituição seguro
if auc_novo > auc_atual:
    joblib.dump(novo_pipeline, PATH_PRODUCAO)
    print("SUCESSO: Novo modelo v2 superou o modelo atual e foi implantado em produção!")
else:
    print("RETENÇÃO: O novo modelo não superou a versão atual. Mantendo modelo v1 em produção.")

**Entendendo as variáveis da substituição segura:**
- `auc_atual`: Score de performance do modelo que já está ativo em produção medido nos dados recentes de validação.
- `auc_novo`: Score de performance alcançado pela nova versão v2 recém-treinada.
- `joblib.dump(novo_pipeline, PATH_PRODUCAO)`: Sobrescreve o arquivo serializado no disco apenas se a condição `auc_novo > auc_atual` for verdadeira.

---

## Conclusão da Aula — Retreinamento Seguro

Um pipeline seguro de retreinamento só promove o novo modelo se ele comprovar superioridade em relação à versão ativa (*Champion vs Challenger*), garantindo estabilidade e governança.